# Composability vs. tractability

CuTe's layout composition $B \circ A$ ("apply $A$ first, then $B$") is defined
whenever a divisibility condition holds between $A$'s modes and $B$'s shape.
`tract` composes *morphisms*, and a flat layout has a morphism exactly when it
is **tractable**. It is natural to conjecture:

> **Conjecture.** If $B \circ A$ exists, then $A$ is tractable.

This notebook refutes the conjecture, dissects pycute's composition algorithm
into an explicit predicate (verified against pycute on random layouts), and
shows that `tract`'s **mutual refinement** is exactly that predicate
strengthened in two independent ways — the second of which *is* tractability.

It is the executable companion of `COMPOSABILITY_NOTES.md` (repo root), which
records the full investigation. Conventions used throughout:

- **$B \circ A$** means apply $A$ first: `pycute.composition(B, A)`, $A$ inner, $B$ outer.
- A flat layout is **tractable** if, after sorting modes by stride, each
  stride divides the next stride·shape product (`is_tractable`); equivalently,
  it is the layout of a tuple morphism. For tractability *as a function*,
  nullify strides of shape-1 modes first (`nullify_trivial_strides`).
- **In-bounds**: $\operatorname{cosize}(A) \le \operatorname{size}(B)$. pycute
  silently permits out-of-bounds composition (truncation/extension through the
  last mode); C++ CuTe would reject it. All claims below are restricted to
  in-bounds pairs.

In [1]:
import random
from pycute import Layout, composition, size
from pycute.layout import _coalesce_z  # coalesce that keeps shape-1 modes' effect

from tract import NestedTuple, mutual_refinement
from tract.backends import pycute as pb


def cosize(L):
    return L(size(L) - 1) + 1 if size(L) > 0 else 0


def in_bounds(A, B):
    return cosize(A) <= size(B)


def pycute_composes(A, B):
    try:
        composition(B, A)
        return True
    except (ValueError, ZeroDivisionError):
        return False


def coalesced_shape(B):
    new_s, _ = _coalesce_z(B.shape, B.stride)
    return new_s if isinstance(new_s, tuple) else (new_s,)


def is_tractable(A):
    return pb.is_tractable(pb.nullify_trivial_strides(pb.flatten_layout(A)))

## 1. The conjecture, and its refutation

Three verified in-bounds counterexamples. In each, $B \circ A$ exists but $A$
is **not** tractable:

| $B$ | $A$ | remark |
|---|---|---|
| `12:1` | `(2,2):(1,3)` | any $B$ coalescing to rank 1 — which includes **every compact layout** — imposes no condition on $A$ beyond bounds |
| `(4,6):(1,8)` | `(2,2):(8,12)` | $B$ tractable *and* non-coalescible; $A$ injective, non-tractable ($12 \nmid 2\cdot 8$) |
| `8:1` | `(2,2):(1,1)` | overlapping (non-injective) $A$ composes with anything large enough |

In [2]:
counterexamples = [
    (Layout((12,), (1,)),  Layout((2, 2), (1, 3))),
    (Layout((4, 6), (1, 8)), Layout((2, 2), (8, 12))),
    (Layout((8,), (1,)),  Layout((2, 2), (1, 1))),
]
for B, A in counterexamples:
    assert in_bounds(A, B)
    print(f"B = {B},  A = {A}")
    print(f"  B∘A = {composition(B, A)},  A tractable: {is_tractable(A)}")

B = (12,):(1,),  A = (2, 2):(1, 3)
  B∘A = (2, 2):(1, 3),  A tractable: False
B = (4, 6):(1, 8),  A = (2, 2):(8, 12)
  B∘A = (2, 2):(16, 24),  A tractable: False
B = (8,):(1,),  A = (2, 2):(1, 1)
  B∘A = (2, 2):(1, 1),  A tractable: False


**Sharper negative result.** Even $A$ and $B$ *both tractable* with
$B \circ A$ defined does not make $B \circ A$ tractable:

In [3]:
A = Layout((4,), (3,))          # tractable
B = Layout((6, 3), (21, 7))     # tractable
BA = composition(B, A)
print(f"A tractable: {is_tractable(A)},  B tractable: {is_tractable(B)}")
print(f"B∘A = {BA},  tractable: {is_tractable(BA)}")

A tractable: True,  B tractable: True
B∘A = ((2, 2),):((63, 7),),  tractable: False


`tract` diagnoses why. The standard morphisms are
$f_A : (4) \to (3, 4)$ and $f_B : (6,3) \to (7,3,6)$, and composing them
weakly requires mutually refining $\operatorname{cod}(f_A) = (3,4)$ with
$\operatorname{dom}(f_B) = (6,3)$ — which **fails**: the prefix products 12
and 18 are divisibility-incomparable.

In [4]:
fA = pb.compute_Tuple_morphism(A)
fB = pb.compute_Tuple_morphism(B)
print("f_A:", fA)
print("f_B:", fB)
try:
    mutual_refinement(NestedTuple(fA.codomain), NestedTuple(fB.domain))
except ValueError as e:
    print("mutual_refinement((3,4), (6,3)) fails:", e)

f_A: (4,) --(2,)--> (3, 4)
f_B: (6, 3) --(3, 2)--> (7, 3, 6)
mutual_refinement((3,4), (6,3)) fails: The given nested tuples are not mutually refinable.


So CuTe composability is strictly weaker than tract weak-composability,
and tractable layouts are **not** closed under raw CuTe composition. The rest
of the notebook locates the exact gap.

## 2. Anatomy of pycute's composition

pycute's `Layout._composition` reduces everything to one primitive:

1. It **distributes over $A$'s modes**: $B \circ (A_1, A_2, \dots) =
   (B \circ A_1,\, B \circ A_2, \dots)$ — with **no cross-mode condition whatsoever**.
2. It **coalesces $B$**, so only $M := \operatorname{shape}(\operatorname{coalesce_z}(B))
   = (M_1, \dots, M_m)$ matters for existence.
3. It handles the primitive $B \circ (s{:}d)$ in two stages, with exactly two
   `raise` sites.

Write $P_j = M_1 M_2 \cdots M_j$ (with $P_0 = 1$) for the prefix products —
a divisibility chain marking the boundaries of $B$'s mixed-radix coordinate
box. $B \circ (s{:}d)$ walks the arithmetic progression
$0, d, 2d, \dots, (s{-}1)d$ through that box, and the two stages check that
the walk respects the box:

- **Stage 1 (shape condition, on the extent $E = s \cdot d$).** The footprint
  must consume whole modes then stop: divisibility is demanded at every mode
  fully crossed, but the **final** covered mode is exempt (the quotient-0
  truncation branch) — the walk may end ragged.
- **Stage 2 (stride condition, on the step $d$).** Within that footprint, $d$
  must factor through the boundaries it crosses; again the last covered mode
  is exempt (a floor division with no remainder check).

**Slogan: divisibility is required at every internal boundary of
$\operatorname{coalesce}(B)$ that the walk must cross; the walk may *end*
ragged but may never *cross* a boundary misaligned.**

The cell below is a verbatim transcription of the two loops as a pure
predicate.

In [5]:
def mode_composable(s, d, M):
    """Exact transcription of pycute Layout._composition's two loops for a
    single inner mode s:d against M = shape(coalesce_z(B))."""
    if d == 0 or s == 1:
        return True
    # Stage 1: shape condition on the extent E = s*d, walked through M.
    # pycute replaces the last mode's extent by E (extendability), so the
    # last mode is never checked.
    rs = list(M)
    rs[-1] = s * d
    for i in range(len(rs) - 1):
        rs[-1], rES = divmod(rs[-1], rs[i])
        if rs[-1] == 0:
            rs[i] = rES
            rs = rs[: i + 1]
            break
        if rES != 0:
            return False
    # Stage 2: stride condition on the step d, within the covered footprint.
    strideB = d
    for i in range(len(rs) - 1):
        qSD, rSD = divmod(rs[i], strideB)
        if rSD == 0:
            break
        strideB, rDS = divmod(strideB, rs[i])
        if rDS != 0:
            return False
    return True


def composable_pred(A, B):
    """B∘A exists (per-mode over A, against coalesce_z(B))."""
    M = coalesced_shape(B)
    def leaves(x):
        return (l for e in x for l in leaves(e)) if isinstance(x, tuple) else iter((x,))
    return all(
        mode_composable(s, d, M)
        for s, d in zip(leaves(A.shape), leaves(A.stride))
    )

**Verification V1**: the predicate matches `pycute.composition`
success/failure on random in-bounds pairs, with 0 mismatches.

In [6]:
def rand_layout(rng, max_rank=4, max_shape=8):
    r = rng.randint(1, max_rank)
    return Layout(
        tuple(rng.randint(1, max_shape) for _ in range(r)),
        tuple(rng.choice([0, 1, 1, 2, 2, 3, 4, 6, 8, 12, 16, 24]) for _ in range(r)),
    )

rng = random.Random(0)
tried = mismatches = 0
while tried < 50_000:
    A, B = rand_layout(rng), rand_layout(rng)
    if not in_bounds(A, B):
        continue
    tried += 1
    if composable_pred(A, B) != pycute_composes(A, B):
        mismatches += 1
print(f"V1: {tried} random in-bounds pairs, {mismatches} mismatches")
assert mismatches == 0

V1: 50000 random in-bounds pairs, 0 mismatches


## 3. The dictionary to mutual refinement

**Step 1 — a single mode is the two-entry tuple $(d, s)$.** The standard
morphism of the layout $s{:}d$ is $f : (s) \to (d, s)$. tract's greedy
`mutual_refinement((d, s), M)` splits $d$ against $M$ — forcing $d$ to consume
whole modes and *divide* the one it lands in — then splits $s$ across the rest
the same way. These are precisely Stages 2 and 1 above **with the final-mode
exemptions removed**.

**Verification V2**: `mutual_refinement((d, s), M)` succeeding implies pycute
composes, on random in-bounds pairs with 0 exceptions.

In [7]:
def refinable(T, U):
    try:
        mutual_refinement(NestedTuple(T), NestedTuple(U))
        return True
    except ValueError:
        return False

rng = random.Random(1)
tried = violations = 0
while tried < 20_000:
    s, d = rng.randint(2, 12), rng.choice([1, 2, 3, 4, 6, 8, 12])
    A, B = Layout((s,), (d,)), rand_layout(rng)
    if not in_bounds(A, B):
        continue
    tried += 1
    if refinable((d, s), coalesced_shape(B)) and not pycute_composes(A, B):
        violations += 1
print(f"V2: {tried} pairs, {violations} violations of (refinable ⇒ composes)")
assert violations == 0

V2: 20000 pairs, 0 violations of (refinable ⇒ composes)


The gap (pycute yes, mutual refinement no) is exactly the
ragged-final-edge cases. E.g. refining $(3, 4)$ against $(6, 3)$: $d = 3$
divides $M_1 = 6$, but $s \cdot d = 12 = 6 \cdot 2$ ends with $c = 2 \nmid
M_2 = 3$. pycute takes the first two-thirds of the mode; mutual refinement
refuses — and this is exactly where the tractable ∘ tractable →
non-tractable example of §1 lives.

In [8]:
print("refinable((3,4), (6,3)):", refinable((3, 4), (6, 3)))
print("pycute composes (6,3):(21,7) ∘ 4:3:",
      pycute_composes(Layout((4,), (3,)), Layout((6, 3), (21, 7))))

refinable((3,4), (6,3)): False
pycute composes (6,3):(21,7) ∘ 4:3: True


**Step 2 — tractability glues the modes together.** For tractable $A$
in sorted standard form, $T_A = (d_1, s_1, d_2/(s_1 d_1), s_2, \dots)$ has
prefix products $\{1, d_1, s_1 d_1, d_2, s_2 d_2, \dots\}$ — a divisibility
chain, *by tractability*. Mutual refinability of flat tuples amounts to their
prefix-product sets merging into one chain, and merging a chain with the
chain $\{P_j\}$ only requires each element to be comparable with each $P_j$.
Hence the joint condition decomposes per-mode.

**Verification V3**: for random tractable $A$ and arbitrary $B$,
`mutual_refinement(T_A, M)` succeeds **iff** `mutual_refinement((dᵢ, sᵢ), M)`
succeeds for every mode individually.

In [9]:
def rand_tractable(rng):
    m = rng.randint(1, 4)
    modes, bound = [], 1
    for _ in range(m):
        d = bound * rng.choice([1, 1, 2, 3, 4])
        s = rng.randint(2, 6)
        modes.append((s, d))
        bound = s * d
    rng.shuffle(modes)
    return Layout(tuple(s for s, d in modes), tuple(d for s, d in modes))

rng = random.Random(2)
tried = mismatches = 0
while tried < 10_000:
    A, B = rand_tractable(rng), rand_layout(rng)
    if not in_bounds(A, B):
        continue
    tried += 1
    M = coalesced_shape(B)
    T_A = pb.compute_Tuple_morphism(A).codomain
    joint = refinable(T_A, M)
    per_mode = all(
        refinable((d, s), M)
        for s, d in zip(A.shape, A.stride) if s > 1 and d > 0
    )
    if joint != per_mode:
        mismatches += 1
print(f"V3: {tried} pairs, {mismatches} mismatches (joint vs per-mode)")
assert mismatches == 0

V3: 10000 pairs, 0 mismatches (joint vs per-mode)


**Step 3 — the full correspondence.**

| condition | per mode | across $A$'s modes | ragged final edge |
|---|---|---|---|
| pycute composability | boundary divisibility | nothing | allowed |
| tract weak-composability, i.e. `mutual_refinement(T_A, S_B)` | boundary divisibility | chain condition = **tractability of $A$** | forbidden |

So tract's mutual refinement equals pycute's divisibility conditions
strengthened in exactly two independent ways:

1. each mode's walk must **end on a boundary** of the common refinement (no
   truncation), and
2. all modes' walks must be simultaneously compatible with **one** refinement
   of $M$ — forcing the $d_i,\, s_i d_i$ to interleave into a single chain,
   which is precisely tractability of $A$.

The original conjecture fails on (2) alone — per-mode conditions cannot see
cross-mode structure. The tractable ∘ tractable → non-tractable example fails
on (1) alone.

Note that a chain among the multiset $\{d_i, s_i d_i\}$ is *not* sufficient
for tractability: `(2,2):(1,1)` has chain $\{1, 2\}$ yet is non-tractable —
tractability needs the intervals $[d_i, s_i d_i]$ to be disjointly stacked
($d_1 \le s_1 d_1 \mid d_2 \le s_2 d_2 \mid \cdots$), not merely comparable.

**Coalescing caveat.** Everything is stated against
$\operatorname{coalesce_z}(B)$, not $B$: coalescing coarsens the boundary
chain $\{P_j\}$ and strictly weakens the conditions. Mutual refinement
against $B$'s raw shape is conservative:

In [10]:
print("T_A = (3,2) vs raw (4,6):     ", refinable((3, 2), (4, 6)))
print("T_A = (3,2) vs coalesced (24,):", refinable((3, 2), (24,)))

T_A = (3,2) vs raw (4,6):      False
T_A = (3,2) vs coalesced (24,): True


## 4. Candidate theorem statements

1. **Characterization of composability.** For in-bounds $B \circ A$, pycute
   composition exists iff for every mode $s{:}d$ of $A$ (with $s > 1$,
   $d > 0$): $d = P_j \cdot d'$ with $d' \mid M_{j+1}$ (or $d$ lands in the
   final covered mode), and $s \cdot d = P_k \cdot c$ with $c \le M_{k+1}$,
   where $P$ are the prefix products of
   $\operatorname{shape}(\operatorname{coalesce_z}(B))$. *(Verified V1.)*
2. **Strict composability = per-mode mutual refinability.** The
   no-ragged-edge strengthening of (1) holds iff $(d, s)$ and
   $\operatorname{shape}(\operatorname{coalesce_z}(B))$ are mutually
   refinable, for every mode. *(V2.)*
3. **Bridge.** If $A$ is tractable, per-mode mutual refinability is
   equivalent to joint mutual refinability of $T_A$ with
   $\operatorname{shape}(\operatorname{coalesce_z}(B))$, i.e. to tract
   weak-composability of the standard morphisms. *(V3.)*
4. **Closure.** Under (3)'s hypotheses the composite is tractable with
   $f_{B \circ A}$ the weak composite; without them, $B \circ A$ of tractable
   layouts can be non-tractable (§1).

Also relevant to testing against pycute: `make_layout` crashes on empty
iterables, breaking rank-0 edge cases of `composition`, `logical_product`,
and `coalesce` that C++ CuTe handles; `tract.backends.pycute` ships
workaround wrappers (`compose_layouts`, `coalesce_layout`,
`logical_product_layouts`).